# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. должен вывести 20 и 10 потому что iloc выводит индекс 
2. выведет: NaN, Шымкент 3.0, NaN (потому что нет схожих элементов кроме "Шымкент", в добавок есть пару моментов, из-за присутсвия NaN нашк "3" станет float. 2 момент то что он по алфоыитно разложит ответ (алматы->астана->шымкент))
3. (2, 2) выведет так потому что там только 2 элемента но из-за того что это датафрейм то он не убирает "у", без указания конкретного столбца он остаётся полноценной таблицей с двумя столбцами
4. Series DataFrame (в первом 1 скобки а во втором по 2 с каждой строны)
5. выведет - [1, 2] 5  - потому что мы срезали 1 строку по фильрации, остались индексы 1 и 2. а iloc[0, 1] обращается по 1 строке и 2 столбцу отфильтрованого это 5

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [2]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [19]:
# 1
print("строк и столбцов - ", df.shape)

print("типы у столбцов - ", df.dtypes)

print("пропуски в delivery_days - ", df["delivery_days"].isna().sum())

for col in ["city", "category", "status", "customer_id"]:
    print(col, ":", df[col].nunique())

test = ~not_test
print("заказов TEST:", test.sum())

print(df["city"].value_counts())

строк и столбцов -  (5463, 8)
типы у столбцов -  order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
пропуски в delivery_days -  160
city : 14
category : 5
status : 2
customer_id : 2003
заказов TEST: 3
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64


**Три проблемы данных.**
1. в 160 заказах нет срока доставки
2. Клиент TEST: 3 заказа по 9 999 990 ₸, это тестовые данные, они искажают выручку.
3. Города записаны 14 способами («Алматы», «алматы», «Almaty», «АЛМАТЫ» и т. д.) и из-за этого по городам выручка считается неверно


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [22]:
# 2
amounts = df.loc[paid & not_test, "amount"]
print("среднее:", round(amounts.mean()), "| медиана:", amounts.median())
print("25%:", amounts.quantile(0.25), "| 75%:", amounts.quantile(0.75))

среднее: 14672 | медиана: 10300.0
25%: 5880.0 | 75%: 18690.0


Типичный чек около 10 300, большинство заказов от 5 880 до 18 690. В отчёте честнее показывать медиану: среднее (14 672) завышают редкие очень дорогие заказы

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [24]:
# 3
cancel = df["status"] == "отменён"
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

print("все заказы:", round(cancel.mean() * 100, 1), "%")
print("Шымкент:", round(cancel[df["city"].isin(SH)].mean() * 100, 1), "%")
print("Алматы:", round(cancel[df["city"].isin(AL)].mean() * 100, 1), "%")

все заказы: 10.2 %
Шымкент: 20.4 %
Алматы: 7.9 %


В Шымкенте отменяется 20,4% заказов, это вдвое больше, чем в среднем (10,2%), и в 2,6 раза больше, чем в Алматы (7,9%). Шымкент надо проверять в первую очередь

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [25]:
# 4
mask = paid & not_test

for name, names in [("Шымкент", SH), ("Алматы", AL)]:
    m = mask & df["city"].isin(names)
    s = df.loc[m & sep, "amount"].sum()
    o = df.loc[m & octo, "amount"].sum()
    print(f"{name}: {s:,.0f} → {o:,.0f} ₸, {(o / s - 1) * 100:+.1f}%")

Шымкент: 4,949,800 → 4,044,010 ₸, -18.3%
Алматы: 15,523,940 → 14,643,020 ₸, -5.7%


В Шымкенте выручка упала на 18,3%, в Алматы на 5,7%: в Шымкенте падение почти в 3 раза сильнее. Причину падения нужно искать в Шымкенте

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

1) & приоритетнее ==, поэтому python сначала считает "Шымкент" & df["city"] и падает с TypeError. Нужны скобки вокруг каждого условия
2) Тихая ошибка: == "Шымкент" берёт только одно написание и теряет другие значения. Нужен isin(SH)

In [26]:
# 5
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [35]:
# 6
top = df["customer_id"].value_counts()
print(top.head())

best = top.index[0]
orders = df.loc[df["customer_id"] == best, ["order_id", "order_date", "city", "amount", "status"]]
print(orders)
print("повторов order_id:", orders["order_id"].duplicated().sum())

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
повторов order_id: 1


Больше всех заказов у клиента C01818, 11. Среди них есть дубль (order_id 101504 записан дважды), значит, выручку мы считали с задвоенными заказами и  их надо убрать перед отчётом.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [37]:
# 7
q1, q3 = amounts.quantile(0.25), amounts.quantile(0.25)
border = q3 + 1.5 * (q3 - q1)
big = amounts[amounts > border]

print("граница:", border)
print("выбросов:", len(big), "доля:", round(len(big) / len(amounts) * 100, 1), "%")
print(df.loc[big.index, "category"].value_counts())

граница: 5880.0
выбросов: 3674 доля: 75.0 %
category
Одежда         1296
Электроника    1062
Дом             850
Красота         394
Книги            72
Name: count, dtype: int64


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [38]:
# 8
cancel_cnt = df.loc[df["status"] == "отменён", "city"].value_counts()
total_cnt = df["city"].value_counts()

share = (cancel_cnt / total_cnt).sort_values(ascending=False)
print((share * 100).round(1))
print(total_cnt)

city
Shymkent     23.1
Шымкент      21.5
алматы       14.1
астана       13.1
Актобе       10.3
Astana       10.0
Караганда     8.9
Астана        8.5
Алматы        8.0
Атырау        7.5
Almaty        5.8
 Алматы       4.5
шымкент       3.7
АЛМАТЫ        3.2
Name: count, dtype: float64
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64


У трёх написаний Шымкента доля отмен 23,1%, 21,5% и 3,7%, а заказов 39, 711 и 54. Судить о городе по одному написанию нельзя: на «шымкент» всего 54 заказа, и цифра случайная 